## Libraries and General Checks

In [54]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer

# Use a consistent style for all charts in this notebook.
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

print("\n✓ Environment check passed")

Python: 3.14.2
NumPy: 2.5.3
pandas: 3.0.5

✓ Environment check passed


---
## Splitting the CSV into the 'human' and 'ai' labels

In [ ]:
# Load the dataset
df = pd.read_csv('../data/DataSet_AIvsHuman.csv')
df.head()

In [ ]:
#ai csv file
df_ai = df[df["label"].str.lower() == "ai"]
df_ai.to_csv("../data/ai_only_file.csv", index=False)
df_ai.head()

#human csv file
df_human = df[df["label"].str.lower() == "human"]
df_human.to_csv("../data/human_only_file.csv", index=False)
df_human.head()

---
## First Look

In [ ]:
# shape returns (number_of_rows, number_of_columns).
print("rows, columns:", df.shape)
print("rows, columns:", df_ai.shape)
print("rows, columns:", df_human.shape)

# info() shows column names, data types, and non-missing value counts.
print('\nDataFrame Info:')
df.info()


# Count missing cells in each column.
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
})

# Sort so the columns needing the most attention appear first.
missing.sort_values("n_missing", ascending=False)

# duplicated() marks rows that are exact copies of an earlier row.
# sum() counts how many duplicate rows were found.
print("\nExact duplicate rows:", df.duplicated().sum())

# describe() calculates common summaries for numeric columns.
# Transpose so each variable appears as a row, which is easier to read.
df.describe().T

# Select numeric columns and draw one histogram for each.
df.select_dtypes("number").hist(figsize=(10, 6), bins=30, edgecolor="white")
plt.tight_layout()
plt.show()


# Select numeric columns and calculate Pearson correlations.
# Pearson is pandas' default method and measures linear relationships.
numeric_data = df.select_dtypes("number")
corr = numeric_data.corr(method="pearson")

# A heatmap makes positive and negative relationships easier to scan.
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "Pearson correlation"})
plt.title("Pearson correlation matrix")
plt.tight_layout()
plt.show()

# ai data
numeric_data = df_ai.select_dtypes("number")
corr = numeric_data.corr(method="pearson")

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "Pearson correlation"})
plt.title("Pearson correlation matrix")
plt.tight_layout()
plt.show()

# human data
numeric_data = df_human.select_dtypes("number")
corr = numeric_data.corr(method="pearson")

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "Pearson correlation"})
plt.title("Pearson correlation matrix")
plt.tight_layout()
plt.show()

---
## Distribution of data

AI data

In [ ]:
# Compare four ways to show the distribution of one numeric column.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))

sns.histplot(data=df_ai, x="char_count", bins=25, ax=axes[0])
axes[0].set_title("Histogram")

sns.boxplot(data=df_ai, y="char_count", ax=axes[1])
axes[1].set_title("Box plot")

sns.violinplot(data=df_ai, y="char_count", ax=axes[2])
axes[2].set_title("Violin")

sns.ecdfplot(data=df_ai, x="char_count", ax=axes[3])
axes[3].set_title("ECDF")

plt.tight_layout()
plt.show()

---
Human data

In [ ]:
# Compare four ways to show the distribution of one numeric column.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))

sns.histplot(data=df_human, x="char_count", bins=25, ax=axes[0])
axes[0].set_title("Histogram")

sns.boxplot(data=df_human, y="char_count", ax=axes[1])
axes[1].set_title("Box plot")

sns.violinplot(data=df_human, y="char_count", ax=axes[2])
axes[2].set_title("Violin")

sns.ecdfplot(data=df_human, x="char_count", ax=axes[3])
axes[3].set_title("ECDF")

plt.tight_layout()
plt.show()

---
Combined data

In [ ]:
# Compare four ways to show the distribution of one numeric column.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))

sns.histplot(data=df, x="char_count", bins=25, ax=axes[0])
axes[0].set_title("Histogram")

sns.boxplot(data=df, y="char_count", ax=axes[1])
axes[1].set_title("Box plot")

sns.violinplot(data=df, y="char_count", ax=axes[2])
axes[2].set_title("Violin")

sns.ecdfplot(data=df, x="char_count", ax=axes[3])
axes[3].set_title("ECDF")

plt.tight_layout()
plt.show()

---
## Comparisons

AI data

In [ ]:
# Compare a summary plot, a spread plot, and the individual observations.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

sns.barplot(data=df_ai, x="ai_model", y="char_count", errorbar=None, ax=axes[0])
axes[0].set_title("Bar of means: hides everything")

sns.boxplot(data=df_ai, x="ai_model", y="char_count", ax=axes[1])
axes[1].set_title("Box: shows spread")

sns.stripplot(data=df_ai, x="ai_model", y="char_count", alpha=0.5, ax=axes[2])
axes[2].set_title("Every input: shows the truth")

plt.tight_layout()
plt.show()

---
Human data

In [ ]:
# Compare a summary plot, a spread plot, and the individual observations.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

sns.barplot(data=df_human, x="source", y="complexity_score", errorbar=None, ax=axes[0])
axes[0].set_title("Bar of means: hides everything")

sns.boxplot(data=df_human, x="source", y="complexity_score", ax=axes[1])
axes[1].set_title("Box: shows spread")

sns.stripplot(data=df_human, x="source", y="complexity_score", alpha=0.5, ax=axes[2])
axes[2].set_title("Every input: shows the truth")

plt.tight_layout()
plt.show()

---
All data

In [ ]:
# Compare a summary plot, a spread plot, and the individual observations.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

sns.barplot(data=df, x="label", y="char_count", errorbar=None, ax=axes[0])
axes[0].set_title("Bar of means: hides everything")

sns.boxplot(data=df, x="label", y="char_count", ax=axes[1])
axes[1].set_title("Box: shows spread")

sns.stripplot(data=df, x="label", y="char_count", alpha=0.5, ax=axes[2])
axes[2].set_title("Every input: shows the truth")

plt.tight_layout()
plt.show()

---
## Relationship: do two variables move together?

When your question is **“Is X associated with Y?”**, start with a scatter plot.

Here we compare bill length and bill depth. The displayed `r` value is **Pearson correlation**, the default method used by pandas. It measures a straight-line relationship, not every possible kind of association. First look at the overall pattern; then split by species.

In [ ]:
# Remove rows missing either variable before calculating correlations.
sub = df.dropna(subset=["char_count", "word_count"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: the overall relationship across all penguins.
# corr() uses Pearson correlation by default, so the line measures linear association.
sns.regplot(data=sub, x="char_count", y="word_count",
            scatter_kws={"alpha": 0.6}, line_kws={"color": "black"}, ax=axes[0])
axes[0].set_title(f"(Pearson r = {sub['char_count'].corr(method='pearson', other=sub['word_count']):.2f})")

# Right: draw one fitted line for each species.
for sp, g in sub.groupby("label"):
    sns.regplot(data=g, x="char_count", y="word_count",
                scatter_kws={"alpha": 0.6}, label=sp, ax=axes[1])
axes[1].legend(title="label")
axes[1].set_title("By label: every slope flips")

plt.tight_layout()
plt.show()

print("overall: ", round(sub["char_count"].corr(method="pearson", other=sub["word_count"]), 3))
for sp, g in sub.groupby("label"):
    print(f"{sp:12s}", round(g["char_count"].corr(method="pearson", other=g["word_count"]), 3))

---
## Preprocessing & Feature Engineering

Libraries and Impoets for preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer

## Target and features — check for target leakage

We want to predict `label`.

**Think:** for every column, ask *would this be available **before** we know the satisfaction score?*

- Which column only exists **because of** the outcome? That is **target leakage**: remove that column.
- Which column carries no information at all and should not be a feature?

In [ ]:
target = "label"

leaky_cols = ["source", "ai_model"]
id_cols = ["id"]

X = df.drop(columns=[target] + leaky_cols + id_cols)
y = df[target]
print("Features used:", X.columns.tolist())

# X (clean_data.csv) removes the 'label', 'source', 'ai_model', and 'id' columns.

X.to_csv("../data/clean_data.csv", index=False)
X.head()


Features used: ['prompt', 'content', 'type', 'topic', 'word_count', 'char_count', 'language', 'complexity_score', 'is_multiline_code']


,prompt,content,type,topic,word_count,char_count,language,complexity_score,is_multiline_code
0,Write a string manipulation solution in python,def string_manipulation():\n # TODO: implem...,code,string manipulation,9,84,python,3,True
1,Explain health in simple terms,Experts recommend focusing on health to improv...,text,health,29,211,en,3,False
2,Summarize recent developments in finance,The development of finance technologies has ac...,text,finance,56,419,en,5,False
3,Summarize recent developments in finance,Recent studies show that finance can have sign...,text,finance,42,298,en,10,False
4,Provide a graph traversal algorithm in cpp,vector<int> arr; // graph traversal example\nf...,code,graph traversal,11,98,cpp,10,True
